# 🦷 BrightSmile Dental Chatbot — LLaMA Fine-Tuning

**Complete pipeline: Data Loading → Intent Classifier → JSONL → LLaMA Fine-Tuning → Testing → Export**

> **Requirements:** Google Colab with **T4 GPU** selected.
> Go to **Runtime → Change runtime type → T4 GPU** before running.

---

## Step 0: Enter Your Hugging Face Token
Get a free token at https://huggingface.co/settings/tokens (select "Write" permission).
This is needed to download LLaMA and upload your trained model.

In [ ]:
# Your Hugging Face credentials
HF_TOKEN = ""  # Paste your HF token here
HF_USERNAME = ""  # Your HF username

assert HF_TOKEN.startswith("hf_"), "❌ Token error — paste your token above"
assert HF_USERNAME, "❌ Username error — enter your username above"
print(f"✅ Token set for user: {HF_USERNAME}")

## Step 1: Install All Required Libraries
This installs everything automatically — takes about 2-3 minutes.

In [ ]:
%%capture install_output
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q scikit-learn pandas openpyxl huggingface_hub

import torch
print(f"✅ All libraries installed!")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: Upload Your Training Data Files
Click the **Choose Files** button below and select your Excel files.
Upload one or both: `dental_feature_qa_4000.xlsx` and/or `dental_chatbot_qa_final.xlsx`

In [ ]:
from google.colab import files
import pandas as pd

print("📁 Click 'Choose Files' and select your Excel training files...")
print("   You can upload 1 or 2 files — all will be combined.\n")
uploaded = files.upload()

# Load all uploaded Excel files and combine them
frames = []
for filename in uploaded.keys():
    print(f"Loading: {filename}")
    # Handle files that lost their .xlsx extension during upload
    try:
        df_temp = pd.read_excel(filename, engine="openpyxl")
    except Exception:
        # Try reading as openpyxl with explicit BytesIO
        import io
        df_temp = pd.read_excel(io.BytesIO(uploaded[filename]), engine="openpyxl")
    # Normalize column name
    if "Feature Category" in df_temp.columns:
        df_temp = df_temp.rename(columns={"Feature Category": "Category"})
    frames.append(df_temp)
    print(f"  → {len(df_temp)} rows, columns: {list(df_temp.columns)}")

df = pd.concat(frames, ignore_index=True)
df = df.dropna(subset=["Question", "Expected Answer"])
df = df[df["Question"].astype(str).str.strip() != ""]
df = df[df["Expected Answer"].astype(str).str.strip() != ""]

print(f"\n✅ Total training rows loaded: {len(df)}")
print(f"\nCategory distribution:")
print(df["Category"].value_counts().to_string())

## Step 3: Map Intents & Define Engine Routing
Maps every category to an intent and assigns each intent to the correct AI engine.

In [ ]:
import json

# ── Intent map: Category name → intent label ──
intent_map = {
    # From original 2000-row file
    "Appointment Scheduling":             "book_appointment",
    "Doctor Availability":                "check_availability",
    "Dental Services Info":               "services_info",
    "Pricing & Payment":                  "pricing_info",
    "Office Hours & Location":            "office_hours",
    "Oral Hygiene Tips":                  "oral_hygiene",
    "Specialist Recommendations":         "specialist_recommendation",
    "Post-Treatment Care":                "post_treatment",
    "Emergency Dental":                   "emergency",
    "General Dental Questions":           "general_dental",
    # From 4000-row feature file
    "Appointment Booking System":         "book_appointment",
    "Insurance Verification":             "insurance_verification",
    "FAQ & Knowledge Base":               "faq_general",
    "Emergency Triage & Routing":         "emergency",
    "Patient Intake & Digital Forms":     "patient_intake",
    "Contact Leaving & Callbacks":        "contact_callback",
    "Treatment Upselling & Education":    "treatment_education",
    "Patient Recall & Reactivation":      "patient_recall",
    "Payment Plans & Financing":          "payment_financing",
    "Promotions & Campaigns":             "promotions",
    "No-Show Reduction & Reminders":      "noshow_reminders",
    "Multilingual Support":               "multilingual",
    "HIPAA GDPR Compliance":              "compliance",
    "PMS Integration":                    "pms_integration",
    "Analytics Dashboard":                "analytics",
    "Multi-Location & DSO Support":       "multi_location",
}

# ── Engine routing: intent → which AI engine handles it ──
ENGINE_ROUTING = {
    "symptom_detection":         "CHATGPT",
    "book_appointment":          "BOOKING_ENGINE",
    "check_availability":        "CALENDAR_SYSTEM",
    "pricing_info":              "PRICING_DATABASE",
    "payment_financing":         "PRICING_DATABASE",
    "emergency":                 "EMERGENCY_HANDLER",
    "insurance_verification":    "INSURANCE_ENGINE",
    "patient_intake":            "INTAKE_ENGINE",
    "contact_callback":          "CONTACT_ENGINE",
    "noshow_reminders":          "REMINDER_ENGINE",
    "faq_general":               "GROK_AI",
    "services_info":             "GROK_AI",
    "treatment_education":       "GROK_AI",
    "patient_recall":            "GROK_AI",
    "promotions":                "GROK_AI",
    "multilingual":              "GROK_AI",
    "compliance":                "GROK_AI",
    "pms_integration":           "GROK_AI",
    "analytics":                 "GROK_AI",
    "multi_location":            "GROK_AI",
    "office_hours":              "GROK_AI",
    "oral_hygiene":              "GROK_AI",
    "specialist_recommendation": "GROK_AI",
    "post_treatment":            "GROK_AI",
    "general_dental":            "GROK_AI",
}

CLINIC_CONTEXT = """You are the AI assistant for BrightSmile Advanced Dental Center.
Address: King Fahd Road, Al Olaya District, Riyadh 12214, near Kingdom Centre Tower, 3rd Floor.
Phone: +966 11 234 5678 | WhatsApp: +966 55 987 6543 | Emergency 24/7: +966 50 111 2222
Hours: Saturday-Thursday 9AM-10PM | Friday 2PM-10PM | Emergency: 24/7
Doctors: Dr. Ahmed Al-Harbi (General Dentistry, 12 years), Dr. Sarah Al-Qahtani (Orthodontics, 10 years),
         Dr. Mohammed Al-Otaibi (Oral Surgery, 15 years), Dr. Lina Al-Salem (Pediatric Dentistry, 8 years),
         Dr. Khalid Al-Faisal (Cosmetic Dentistry, 11 years)
Prices in USD: Consultation $27 | Cleaning $67 | Filling $53-$107 | Root Canal $213-$400 |
               Extraction $80-$187 | Implant $800-$1,600 | Whitening $267-$480 |
               Braces $2,133-$4,000 | Veneers $320-$667 per tooth
Insurance: Bupa Arabia, Tawuniya, MedGulf, Allianz Saudi Fransi, AXA Cooperative Insurance
Payment: Cash, Visa, MasterCard, Mada, Apple Pay, STC Pay, Tabby, Tamara"""

BLOCKED_RESPONSE = ("I'm sorry, I can only assist with dental-related topics for "
    "BrightSmile Advanced Dental Center. How can I help you with your "
    "dental needs? You can reach us at +966 11 234 5678 or WhatsApp +966 55 987 6543.")

ALLOWED_INTENTS = list(ENGINE_ROUTING.keys())

# Map categories to intents
df["Category"] = df["Category"].astype(str).str.strip()
df["intent"] = df["Category"].map(intent_map)
unmapped = df[df["intent"].isna()]["Category"].unique()
if len(unmapped) > 0:
    print(f"⚠️ Unmapped categories (will be skipped): {unmapped}")
df = df.dropna(subset=["intent"])

print(f"✅ {len(df)} rows mapped to {df['intent'].nunique()} intents\n")
print("Engine routing summary:")
for intent, engine in ENGINE_ROUTING.items():
    count = len(df[df["intent"] == intent])
    if count > 0:
        print(f"  {engine:<25} {intent:<35} {count} rows")

# Save engine routing
with open("engine_routing.json", "w") as f:
    json.dump(ENGINE_ROUTING, f, indent=2)
print("\n✅ engine_routing.json saved")

## Step 4: Train Intent Classifier (Sklearn)
Trains a TF-IDF + Logistic Regression classifier. Target: >85% accuracy.

In [ ]:
import pickle
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df["Question"].astype(str)
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} | Test set: {len(X_test)}\n")

classifier = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=15000,
        sublinear_tf=True,
        min_df=1
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        C=5.0,
        class_weight="balanced"
    ))
])

classifier.fit(X_train, y_train)
predictions = classifier.predict(X_test)

print("=" * 60)
print("INTENT CLASSIFIER RESULTS")
print("=" * 60)
print(classification_report(y_test, predictions))

# Calculate overall accuracy
from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, predictions)
print(f"\n{'✅' if acc > 0.85 else '⚠️'} Overall accuracy: {acc:.1%} (target: >85%)")

# Save classifier
with open("intent_classifier.pkl", "wb") as f:
    pickle.dump(classifier, f)
print("✅ intent_classifier.pkl saved")

## Step 5: Build & Validate JSONL Training File
Creates the fine-tuning dataset in chat format for LLaMA.

In [ ]:
SYSTEM_PROMPT = f"""You are the AI assistant for BrightSmile Advanced Dental Center. You are strictly domain-locked to dental topics only. Never answer anything outside dentistry or clinic operations. Always use real BrightSmile data. {CLINIC_CONTEXT} For off-topic questions respond only with: {BLOCKED_RESPONSE}"""

entries = []
skipped = 0

for _, row in df.iterrows():
    question = str(row.get("Question", "")).strip()
    answer = str(row.get("Expected Answer", "")).strip()
    intent = str(row.get("intent", "")).strip()
    if not question or not answer or question == "nan" or answer == "nan":
        skipped += 1
        continue
    engine = ENGINE_ROUTING.get(intent, "GROK_AI")
    entries.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ],
        "metadata": {
            "intent": intent,
            "engine": engine,
            "category": str(row.get("Category", ""))
        }
    })

jsonl_path = "brightsmile_training.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as f:
    for entry in entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
print(f"✅ JSONL created: {len(entries)} entries, {skipped} skipped")

# Validate
valid = 0
invalid = 0
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
            msgs = obj.get("messages", [])
            roles = [m["role"] for m in msgs]
            contents = [m["content"] for m in msgs]
            if (all(r in roles for r in ["system", "user", "assistant"])
                    and all(c.strip() for c in contents)):
                valid += 1
            else:
                invalid += 1
        except Exception:
            invalid += 1

print(f"✅ Validation: {valid} valid, {invalid} invalid")
assert invalid == 0, "❌ Fix invalid JSONL entries before proceeding"
print("✅ All entries valid — ready for fine-tuning!")

## Step 6: Load LLaMA 3.2 1B Model (4-bit Quantized)
Downloads the model from Hugging Face (~700 MB). Takes about 2-3 minutes.

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Prepare dataset from JSONL
texts = []
with open("brightsmile_training.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        msgs = obj["messages"]
        text = (
            f"<|system|>\n{msgs[0]['content']}\n"
            f"<|user|>\n{msgs[1]['content']}\n"
            f"<|assistant|>\n{msgs[2]['content']}"
        )
        texts.append({"text": text})

dataset = Dataset.from_list(texts)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"✅ Dataset ready — Train: {len(dataset['train'])} | Validation: {len(dataset['test'])}")

# Load model in 4-bit quantization (fits in T4 GPU memory)
model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"\n⏳ Downloading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

print(f"✅ LLaMA model loaded on {model.device}")
print(f"   Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

## Step 7: Apply LoRA Adapters & Fine-Tune
Adds lightweight LoRA adapters and trains for 3 epochs. Takes ~30-60 minutes on T4 GPU.
You'll see a clear progress bar with ETA.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Apply LoRA — only trains ~2% of parameters (very efficient)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenize dataset — labels = input_ids for causal LM
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_ds = dataset["train"].map(tokenize_fn, batched=True, remove_columns=["text"])
eval_ds = dataset["test"].map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

# Training configuration optimized for T4 GPU (15GB VRAM)
training_args = TrainingArguments(
    output_dir="./brightsmile-llama",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    save_steps=200,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="paged_adamw_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

print("\n🚀 Starting BrightSmile fine-tuning...")
print(f"   Training samples: {len(train_ds)}")
print(f"   Validation samples: {len(eval_ds)}")
print(f"   Epochs: 3 | Batch size: 4 | Grad accumulation: 4")
print(f"   Effective batch size: {4 * 4} = 16")
print(f"   Estimated time: 30-60 minutes on T4 GPU\n")

trainer.train()

print("\n✅ Training complete!")

## Step 8: Save Trained Model

In [ ]:
import shutil, os

output_dir = "./brightsmile-llama-final"
os.makedirs(output_dir, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Copy supporting files into the model folder
for fname in ["intent_classifier.pkl", "brightsmile_training.jsonl", "engine_routing.json"]:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(output_dir, fname))

print("✅ Model saved to ./brightsmile-llama-final/")
print("\nFiles in model folder:")
for f in sorted(os.listdir(output_dir)):
    size = os.path.getsize(os.path.join(output_dir, f))
    if size >= 1e6:
        print(f"  {f:<45} {size/1e6:.1f} MB")
    else:
        print(f"  {f:<45} {size/1024:.0f} KB")

## Step 9: Run All 16 Integration Tests
Tests every engine is correctly triggered by the intent classifier.

In [ ]:
# Load the trained classifier for testing
with open("intent_classifier.pkl", "rb") as f:
    clf = pickle.load(f)

# Keyword fallback (mirrors production smart_router)
_EMERGENCY_KW = ["swollen","swelling","bleeding","broken","knocked out","abscess",
    "severe pain","unbearable","jaw swollen","face swollen","pus","fever","emergency",
    "hurt","hurts","pain","ache","sore","throbbing","toothache"]
_PRICING_KW = ["cost","price","how much","pricing","fee","charge","installment",
    "financing","payment plan","how much does"]
_INSURANCE_KW = ["insurance","bupa","tawuniya","medgulf","allianz","axa",
    "coverage","covered","member id","accept my insurance"]
_BOOKING_KW = ["book","booking","appointment","schedule","reschedule","cancel appointment"]
_INTAKE_KW = ["medical history","intake form","registration form","consent form",
    "before my first visit","fill out form"]
_CALLBACK_KW = ["call me","call back","callback","contact me","leave my number",
    "not ready to book","want a doctor to call"]
_REMINDER_KW = ["remind me","reminder","confirm my appointment","no-show",
    "missed appointment","missed my appointment","rebook"]

def detect_keyword(msg):
    lower = msg.lower()
    if any(kw in lower for kw in _EMERGENCY_KW): return "emergency"
    if any(kw in lower for kw in _INSURANCE_KW): return "insurance_verification"
    if any(kw in lower for kw in _PRICING_KW): return "pricing_info"
    if any(kw in lower for kw in _INTAKE_KW): return "patient_intake"
    if any(kw in lower for kw in _CALLBACK_KW): return "contact_callback"
    if any(kw in lower for kw in _REMINDER_KW): return "noshow_reminders"
    if any(kw in lower for kw in _BOOKING_KW): return "book_appointment"
    return None

# Off-topic detection
_OFF_TOPIC_KW = ["weather","football","soccer","joke","funny","riddle","movie",
    "restaurant","recipe","cook","stock","bitcoin","politics","code","programming"]
_DENTAL_KW = ["tooth","teeth","dental","dentist","doctor","appointment","pain",
    "implant","braces","whitening","cleaning","extraction","gum","jaw","oral",
    "clinic","price","cost","insurance","emergency","smile","filling","crown",
    "bridge","veneer","surgery","x-ray","checkup","consultation","intake","form",
    "medical history","callback","call me","reminder","no-show","recall","promotion",
    "financing","compliance","privacy","analytics","dashboard","pms","multilingual",
    "arabic","bupa","tawuniya","medgulf","brightsmile","open","close","hours"]

def is_off_topic(msg, intent, conf):
    lower = msg.lower()
    if intent and conf > 0.45 and intent in ALLOWED_INTENTS: return False
    if any(kw in lower for kw in _DENTAL_KW): return False
    if any(kw in lower for kw in _OFF_TOPIC_KW): return True
    if conf < 0.3: return True
    return False

def handle_message(msg):
    intent = clf.predict([msg])[0]
    probs = clf.predict_proba([msg])[0]
    conf = max(probs)
    if is_off_topic(msg, intent, conf):
        return "blocked", "RESTRICTION_FILTER", conf
    kw = detect_keyword(msg)
    effective = intent
    if not intent or conf < 0.4:
        if kw: effective = kw
    else:
        if kw in ("emergency","insurance_verification","pricing_info","noshow_reminders") and intent != kw:
            effective = kw
    engine = ENGINE_ROUTING.get(effective, "GROK_AI")
    return effective, engine, conf

# ── 16 Test Cases ──
test_cases = [
    ("Symptom Detection",       "my tooth hurts very bad and my face is swollen",          "EMERGENCY_HANDLER"),
    ("Booking",                  "i want to book an appointment with dr sarah tomorrow",   "BOOKING_ENGINE"),
    ("Availability",             "is dr mohammed available this week",                      "CALENDAR_SYSTEM"),
    ("Pricing",                  "how much does an implant cost at BrightSmile",            "PRICING_DATABASE"),
    ("Emergency",                "i have severe bleeding that wont stop after extraction",  "EMERGENCY_HANDLER"),
    ("Insurance",                "does BrightSmile accept my Tawuniya insurance",           "INSURANCE_ENGINE"),
    ("Intake",                   "i need to fill my medical history before my first visit", "INTAKE_ENGINE"),
    ("Callback",                 "i am not ready to book but want a doctor to call me",    "CONTACT_ENGINE"),
    ("Reminder",                 "did i confirm my appointment tomorrow",                   "REMINDER_ENGINE"),
    ("Grok - Oral Hygiene",      "how do i clean around my new implant",                   "GROK_AI"),
    ("Grok - Services",          "what is a Hollywood smile makeover",                      "GROK_AI"),
    ("Grok - Recall",            "i havent visited in 9 months should i come in",           "GROK_AI"),
    ("Grok - Compliance",        "how does BrightSmile protect my personal data",           "GROK_AI"),
    ("Off-topic Block 1",        "what is the weather in Riyadh",                           "RESTRICTION_FILTER"),
    ("Off-topic Block 2",        "tell me a joke",                                          "RESTRICTION_FILTER"),
    ("Off-topic Block 3",        "recommend a good restaurant near Kingdom Centre",         "RESTRICTION_FILTER"),
]

print("=" * 100)
print("FULL SYSTEM INTEGRATION TEST — 16 Test Cases")
print("=" * 100)
print(f"\n{'Test':<25} {'Intent':<30} {'Engine':<25} {'Status'}")
print("-" * 100)

all_passed = True
for label, message, expected in test_cases:
    intent, engine, conf = handle_message(message)
    passed = engine == expected
    if not passed: all_passed = False
    status = "✅ PASS" if passed else f"❌ FAIL (got {engine})"
    print(f"{label:<25} {intent:<30} {engine:<25} {status}")

print(f"\n{'='*100}")
if all_passed:
    print("🎉 ALL 16 TESTS PASSED!")
else:
    print("⚠️ SOME TESTS FAILED — review above")
print("=" * 100)

## Step 10: Upload to Hugging Face
Pushes the trained model to your Hugging Face account so you can use it anywhere.

In [ ]:
from huggingface_hub import HfApi, create_repo

repo_name = f"{HF_USERNAME}/brightsmile-dental-chatbot"

print(f"⏳ Uploading model to Hugging Face: {repo_name}")
print("   This may take a few minutes...\n")

# Create repo if it doesn't exist
try:
    create_repo(repo_name, token=HF_TOKEN, exist_ok=True)
except Exception as e:
    print(f"  Repo note: {e}")

# Upload entire model folder
api = HfApi()
api.upload_folder(
    folder_path="./brightsmile-llama-final",
    repo_id=repo_name,
    token=HF_TOKEN,
)

print(f"\n✅ Model uploaded to: https://huggingface.co/{repo_name}")

## Step 11: Download Trained Files to Your Computer
Click the download links that appear below to save files to your Mac.

In [ ]:
from google.colab import files

print("📥 Downloading files to your computer...\n")

for fname in ["intent_classifier.pkl", "engine_routing.json", "brightsmile_training.jsonl"]:
    try:
        files.download(fname)
        print(f"  ✅ {fname}")
    except Exception as e:
        print(f"  ⚠️ {fname}: {e}")

print(f"\n🎉 ALL DONE!")
print("=" * 60)
print("SUMMARY:")
print(f"  • Intent classifier trained on {len(df)} rows")
print(f"  • LLaMA model fine-tuned and uploaded to Hugging Face")
print(f"  • Engine routing: {len(ENGINE_ROUTING)} intents across 10 engines")
print(f"  • Hugging Face: https://huggingface.co/{repo_name}")
print("=" * 60)
print("\nCopy intent_classifier.pkl to your microservice/ folder to use it.")